<a href="https://colab.research.google.com/github/NatanaelAnantaSatrio/SistemRekomendasi/blob/main/Praktikum1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import os
from collections import Counter
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
tags_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SistemRekomendasi/tag.csv')
print(tags_df.head)

<bound method NDFrame.head of         userId  movieId            tag            timestamp
0           18     4141    Mark Waters  2009-04-24 18:19:40
1           65      208      dark hero  2013-05-10 01:41:18
2           65      353      dark hero  2013-05-10 01:41:19
3           65      521  noir thriller  2013-05-10 01:39:43
4           65      592      dark hero  2013-05-10 01:41:18
...        ...      ...            ...                  ...
465559  138446    55999        dragged  2013-01-23 23:29:32
465560  138446    55999  Jason Bateman  2013-01-23 23:29:38
465561  138446    55999         quirky  2013-01-23 23:29:38
465562  138446    55999            sad  2013-01-23 23:29:32
465563  138472      923  rise to power  2007-11-02 21:12:47

[465564 rows x 4 columns]>


**Preprocessing**

Di sel ini, kita mengubah semua menjadi huruf, menghapus spasi yang tidak perlu, dan mengelompokkan tag berdasarkan film. Dengan cara ini, setiap movieid akan memiliki daftar dengan semua tag yang terkait, yang mempermudah pemerosesan selanjutnya


In [ ]:
tags_df['tag'] = tags_df['tag'].str.lower().str.strip()
movie_tags_df = tags_df.groupby('movieId')['tag'].apply(list).reset_index()
print(movie_tags_df.head())

   movieId                                                tag
0        1  [watched, computer animation, disney animated ...
1        2  [time travel, adapted from:book, board game, c...
2        3  [old people that is actually funny, sequel fev...
3        4  [chick flick, revenge, characters, chick flick...
4        5  [diane keaton, family, sequel, steve martin, w...


In [ ]:
print(tags_df.tail())

        userId  movieId            tag            timestamp
465559  138446    55999        dragged  2013-01-23 23:29:32
465560  138446    55999  jason bateman  2013-01-23 23:29:38
465561  138446    55999         quirky  2013-01-23 23:29:38
465562  138446    55999            sad  2013-01-23 23:29:32
465563  138472      923  rise to power  2007-11-02 21:12:47


In [ ]:
from pandas.tseries import frequencies
from re import T
movie_tags_df['tag'] = movie_tags_df['tag'].apply(lambda x: x if isinstance(x, list) else [])
movie_tags_df['tag'] = movie_tags_df['tag'].apply(lambda x: [t for t in x if isinstance(t, str) and len(t.split()) == 5])
tags_count = Counter(tag for tags_list in movie_tags_df['tag'] for tag in tags_list)
min_films = 3
frequent_tags = [tag for tag, count in tags_count.items() if count >= min_films]
movie_tags_df['tag'] = movie_tags_df['tag'].apply(lambda x: [t for t in x if t in frequent_tags])
print(f"jumlah total tag setelah penyaringan: {len(frequent_tags)}")

jumlah total tag setelah penyaringan: 147


In [ ]:
D = len(movie_tags_df)
tag_document_count = Counter(tag for tags_list in movie_tags_df['tag'] for tag in tags_list)
tag_idf = {tag: np.log(D / count) for tag, count in tag_document_count.items()}

In [ ]:
all_tags = list(tag_idf.keys())
movie_tags_matrix = pd.DataFrame(0.0, index=movie_tags_df['movieId'], columns=all_tags)
for idx, row in movie_tags_df.iterrows():
    movie_id = row['movieId']
    tags_list = set(row['tag'])
    for tag in tags_list:
      tf = 1
      idf = tag_idf[tag]
      movie_tags_matrix.at[movie_id, tag] = tf * idf

In [ ]:
print("tag yang paling relevan untuk film ini:")
print(movie_tags_matrix.head())

tag yang paling relevan untuk film ini:
         2009 reissue in stereoscopic 3-d  \
movieId                                     
1                                8.781862   
2                                0.000000   
3                                0.000000   
4                                0.000000   
5                                0.000000   

         saturn award (best special effects)  \
movieId                                        
1                                   0.000000   
2                                   6.446488   
3                                   0.000000   
4                                   0.000000   
5                                   0.000000   

         saturn award (best supporting actress)  \
movieId                                           
1                                      0.000000   
2                                      7.801033   
3                                      0.000000   
4                                      0.000000   
5